In [3]:
import json, base64, subprocess, os

os.chdir("D:\\Project_2026\\data-science-learning-journey")
os.makedirs("docs", exist_ok=True)

notebooks_html = """<!DOCTYPE html>
<html>
<head>
    <title>Praveen's Data Science Learning Journey</title>
    <style>
        body { font-family: Arial, sans-serif; max-width: 800px; margin: 50px auto; padding: 20px; }
        h1 { color: #2c3e50; }
        h2 { color: #27ae60; margin-top: 30px; }
        ul { list-style: none; padding: 0; }
        li { margin: 10px 0; }
        a { text-decoration: none; color: #2980b9; font-size: 18px; }
        a:hover { text-decoration: underline; }
    </style>
</head>
<body>
    <h1>Praveen's Data Science Learning Journey</h1>
    
    <h2>Python</h2>
    <ul>
        <li><a href="1.01-Syntax-Semantics-Operators.html">1.01 Syntax, Semantics and Operators</a></li>
        <li><a href="1.02-Control-Flow-Data-Structures-Functions.html">1.02 Control Flow, Data Structures and Functions</a></li>
        <li><a href="1.03-Object-Oriented-Programming-Design.html">1.03 Object Oriented Programming</a></li>
        <li><a href="1.04-Advanced-Modules-Libraries.html">1.04 Advanced Modules and Libraries</a></li>
        <li><a href="1.05-Excercises.html">1.05 Exercises</a></li>
    </ul>

    <h2>Data Structures and Algorithms</h2>
    <ul>
        <li><a href="2.01-Data-Structure-Overview.html">2.01 Data Structure Overview</a></li>
        <li><a href="2.02-The-Big-O-Notion-Recursion.html">2.02 Big O Notation and Recursion</a></li>
        <li><a href="2.03-Arrays-HashTables.html">2.03 Arrays and Hash Tables</a></li>
        <li><a href="2.04-Singly-Linked-Lists.html">2.04 Singly Linked Lists</a></li>
        <li><a href="2.05-Doubly-Linked-Lists.html">2.05 Doubly Linked Lists</a></li>
        <li><a href="2.06-Stacks-Queues.html">2.06 Stacks and Queues</a></li>
        <li><a href="2.07-Binary-Search-Trees.html">2.07 Binary Search Trees</a></li>
        <li><a href="2.08-Tree-Traversal.html">2.08 Tree Traversal</a></li>
        <li><a href="2.09-Heaps.html">2.09 Heaps</a></li>
        <li><a href="2.10-Graphs.html">2.10 Graphs</a></li>
        <li><a href="2.11-Sorting.html">2.11 Sorting</a></li>
    </ul>
</body>
</html>"""

with open("docs\\index.html", "w", encoding="utf-8") as f:
    f.write(notebooks_html)
print("Index page created!")

# Convert Python notebooks
result = subprocess.run(
    "jupyter nbconvert --to html --output-dir docs Topics\\1.0-Python\\*.ipynb",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Convert DSA notebooks
result = subprocess.run(
    "jupyter nbconvert --to html --output-dir docs Topics\\2.0-Data_Structures\\*.ipynb",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Push
for cmd in ["git add .", 'git commit -m "full site"', "git push"]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout or r.stderr)

Index page created!
[NbConvertApp] Converting notebook Topics\1.0-Python\1.01-Syntax-Semantics-Operators.ipynb to html
[NbConvertApp] Writing 554046 bytes to docs\1.01-Syntax-Semantics-Operators.html
[NbConvertApp] Converting notebook Topics\1.0-Python\1.02-Control-Flow-Data-Structures-Functions.ipynb to html
[NbConvertApp] Writing 645412 bytes to docs\1.02-Control-Flow-Data-Structures-Functions.html
[NbConvertApp] Converting notebook Topics\1.0-Python\1.03-Object-Oriented-Programming-Design.ipynb to html
[NbConvertApp] Writing 654437 bytes to docs\1.03-Object-Oriented-Programming-Design.html
[NbConvertApp] Converting notebook Topics\1.0-Python\1.04-Advanced-Modules-Libraries.ipynb to html
[NbConvertApp] Writing 606150 bytes to docs\1.04-Advanced-Modules-Libraries.html
[NbConvertApp] Converting notebook Topics\1.0-Python\1.05-Excercises.ipynb to html
[NbConvertApp] Writing 297603 bytes to docs\1.05-Excercises.html

[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.01-Data

In [5]:
import json, base64, os, subprocess

os.chdir("D:\\Project_2026\\data-science-learning-journey")

dsa_folder = "Topics\\2.0-Data_Structures"
images_folder = "Topics\\2.0-Data_Structures\\DS_Images"

# Process every DSA notebook
for notebook_file in os.listdir(dsa_folder):
    if not notebook_file.endswith(".ipynb"):
        continue
    
    nb_path = os.path.join(dsa_folder, notebook_file)
    
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)
    
    changed = False
    for cell in nb["cells"]:
        if cell["cell_type"] != "markdown":
            continue
        src = "".join(cell["source"])
        if "DS_Images/" not in src and "DS_Images\\" not in src:
            continue
        
        # Find all image references in this cell
        import re
        images = re.findall(r'!\[.*?\]\((?:DS_Images[/\\])(.+?)\)', src)
        
        if not images:
            continue
            
        attachments = {}
        new_src = src
        for img_name in images:
            img_path = os.path.join(images_folder, img_name)
            if not os.path.exists(img_path):
                print(f"  Missing: {img_path}")
                continue
            with open(img_path, "rb") as f:
                img_base64 = base64.b64encode(f.read()).decode()
            attachments[img_name] = {"image/png": img_base64}
            new_src = new_src.replace(f"DS_Images/{img_name}", f"attachment:{img_name}")
            new_src = new_src.replace(f"DS_Images\\{img_name}", f"attachment:{img_name}")
        
        cell["attachments"] = attachments
        cell["source"] = [new_src]
        changed = True
    
    if changed:
        with open(nb_path, "w", encoding="utf-8") as f:
            json.dump(nb, f, indent=1)
        print(f"✅ Patched: {notebook_file}")
    else:
        print(f"⏭ Skipped: {notebook_file}")

print("Done!")

✅ Patched: 2.01-Data-Structure-Overview.ipynb
✅ Patched: 2.02-The-Big-O-Notion-Recursion.ipynb
✅ Patched: 2.03-Arrays-HashTables.ipynb
✅ Patched: 2.04-Singly-Linked-Lists.ipynb
✅ Patched: 2.05-Doubly-Linked-Lists.ipynb
✅ Patched: 2.06-Stacks-Queues.ipynb
✅ Patched: 2.07-Binary-Search-Trees.ipynb
✅ Patched: 2.08-Tree-Traversal.ipynb
✅ Patched: 2.09-Heaps.ipynb
✅ Patched: 2.10-Graphs.ipynb
✅ Patched: 2.11-Sorting.ipynb
⏭ Skipped: Data_Structures_Algorithms_Codes.ipynb
✅ Patched: Github_test.ipynb
✅ Patched: Github_test1.ipynb
✅ Patched: Github_test2.ipynb
✅ Patched: Github_test3.ipynb
✅ Patched: Github_test4.ipynb
Done!


In [7]:
import subprocess, os

os.chdir("D:\\Project_2026\\data-science-learning-journey")

# Reconvert DSA notebooks to HTML
result = subprocess.run(
    "jupyter nbconvert --to html --output-dir docs Topics\\2.0-Data_Structures\\*.ipynb",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Push
for cmd in ["git add .", 'git commit -m "embed all DSA images"', "git push"]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout or r.stderr)

[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.01-Data-Structure-Overview.ipynb to html
[NbConvertApp] Writing 1218488 bytes to docs\2.01-Data-Structure-Overview.html
[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.02-The-Big-O-Notion-Recursion.ipynb to html
[NbConvertApp] Writing 838223 bytes to docs\2.02-The-Big-O-Notion-Recursion.html
[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.03-Arrays-HashTables.ipynb to html
[NbConvertApp] Writing 602125 bytes to docs\2.03-Arrays-HashTables.html
[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.04-Singly-Linked-Lists.ipynb to html
[NbConvertApp] Writing 1336384 bytes to docs\2.04-Singly-Linked-Lists.html
[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.05-Doubly-Linked-Lists.ipynb to html
[NbConvertApp] Writing 946175 bytes to docs\2.05-Doubly-Linked-Lists.html
[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.06-Stacks-Queues.ipynb to html
[NbConvert

In [9]:
import subprocess, os

os.chdir("D:\\Project_2026\\data-science-learning-journey")

result = subprocess.run(
    "jupyter nbconvert --to html --output-dir docs Topics\\2.0-Data_Structures\\2.01-Data-Structure-Overview.ipynb",
    shell=True, capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)


[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.01-Data-Structure-Overview.ipynb to html
[NbConvertApp] Writing 1218488 bytes to docs\2.01-Data-Structure-Overview.html



In [11]:
import subprocess, os

os.chdir("D:\\Project_2026\\data-science-learning-journey")

for cmd in ["git add docs\\2.01-Data-Structure-Overview.html", 
            'git commit -m "fix 2.01 html"', 
            "git push"]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout or r.stderr)


On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   setup.ipynb

no changes added to commit (use "git add" and/or "git commit -a")

Everything up-to-date



In [13]:
import subprocess, os

os.chdir("D:\\Project_2026\\data-science-learning-journey")

# Reconvert only 2.01
result = subprocess.run(
    "jupyter nbconvert --to html --output-dir docs Topics\\2.0-Data_Structures\\2.01-Data-Structure-Overview.ipynb",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Force add and push
for cmd in ["git add -f docs\\2.01-Data-Structure-Overview.html",
            'git commit -m "reconvert 2.01 with images"',
            "git push"]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout or r.stderr)

[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.01-Data-Structure-Overview.ipynb to html
[NbConvertApp] Writing 1218488 bytes to docs\2.01-Data-Structure-Overview.html


On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   setup.ipynb

no changes added to commit (use "git add" and/or "git commit -a")

Everything up-to-date



In [15]:
import subprocess, os

os.chdir("D:\\Project_2026\\data-science-learning-journey")

# Delete old HTML
os.remove("docs\\2.01-Data-Structure-Overview.html")

# Reconvert fresh
result = subprocess.run(
    "jupyter nbconvert --to html --output-dir docs Topics\\2.0-Data_Structures\\2.01-Data-Structure-Overview.ipynb",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Push
for cmd in ["git add docs\\2.01-Data-Structure-Overview.html",
            'git commit -m "fresh 2.01 html with images"',
            "git push"]:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout or r.stderr)

[NbConvertApp] Converting notebook Topics\2.0-Data_Structures\2.01-Data-Structure-Overview.ipynb to html
[NbConvertApp] Writing 1218488 bytes to docs\2.01-Data-Structure-Overview.html


On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   setup.ipynb

no changes added to commit (use "git add" and/or "git commit -a")

Everything up-to-date

